<a href="https://colab.research.google.com/github/stephenebert/Springboard/blob/main/Mini_Project_Building_a_Recommendation_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import zipfile
import numpy as np
import pandas as pd
from urllib.request import urlretrieve
from sklearn.metrics.pairwise import cosine_similarity

In [29]:
print("Downloading movielens data...")

urlretrieve('http://files.grouplens.org/datasets/movielens/ml-100k.zip', 'movielens.zip')
zip_ref = zipfile.ZipFile('movielens.zip', 'r')
zip_ref.extractall()
print("Done. Dataset contains:")
print(zip_ref.read('ml-100k/u.info'))

ratings_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
ratings = pd.read_csv(
    'ml-100k/u.data', sep='\t', names=ratings_cols, encoding='latin-1')

# The movies file contains a binary feature for each genre.
genre_cols = [
    "genre_unknown", "Action", "Adventure", "Animation", "Children", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]
movies_cols = [
    'movie_id', 'title', 'release_date', "video_release_date", "imdb_url"
] + genre_cols
movies = pd.read_csv(
    'ml-100k/u.item', sep='|', names=movies_cols, encoding='latin-1')

Done. Dataset contains:
b'943 users\n1682 items\n100000 ratings\n'


# Part 1

**1. Spend some time familiarizing yourself with both the movies and ratings dataframes. How many unique user ids are present? How many unique movies are there?**

In [37]:
unique_users = ratings['user_id'].nunique()

In [38]:
print(f'{unique_users} are the number of unique users.')

943 are the number of unique users.


In [39]:
unique_movies_ratings = ratings['movie_id'].nunique()

In [40]:
print(f'{unique_movies_ratings} are the number of unique movies.')

1682 are the number of unique movies.


In [41]:
unique_movies_meta = movies['movie_id'].nunique()

In [42]:
print(f"{unique_movies_meta} are the number of unique movies in the movies.")

1682 are the number of unique movies in the movies.


**2. Create a new dataframe that merges the movies and ratings tables on 'movie_id'. Only keep the 'user_id', 'title', 'rating' fields in this new dataframe.**

In [43]:
merged = ratings.merge(movies[['movie_id', 'title']], on = 'movie_id', how='inner')
user_movie_rating = merged[['user_id', 'title', 'rating']]
print(user_movie_rating.head())
print(f'Shape: {user_movie_rating.shape}')

   user_id                       title  rating
0      196                Kolya (1996)       3
1      186    L.A. Confidential (1997)       3
2       22         Heavyweights (1994)       1
3      244  Legends of the Fall (1994)       2
4      166         Jackie Brown (1997)       1
Shape: (100000, 3)


# Part 2

**1. Write a function that takes in a user id and the dataframe you created before that contains 'user_id', 'title', and 'rating'. The function should return content-based recommendations for this user. Here are steps you can take:**

1.   Get the user's rated movies
2.   Create a TF-IDF matrix using movie genres. Note, this can be extracted from the movies dataframe.
3. Compute the cosine similarity between movie genres. Use the cosine_similarity function.
4. Get the indices of similar movies to those rated by the user based on cosine similarity. Keep only the top 5.
5. Remove duplicates and movies already rated by the user.



In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def content_based_recommendations(user_id, user_movie_ratings, movies, genre_cols, top_n = 5):
  # 1. Get the user's rated movies
  rated = user_movie_ratings.loc[user_movie_ratings['user_id']== user_id, 'title'].tolist()

  # 2. Create a TF-IDF matrix using movie genres. Note, this can be extracted from the movies dataframe.
  genre_text = movies[genre_cols].apply(lambda row: ' '.join([g for g in genre_cols if row[g] == 1]), axis = 1)

  # 3. Compute the cosine similarity between movie genres. Use the cosine_similarity function.
  tfidf = TfidfVectorizer()
  tfidf_matrix = tfidf.fit_transform(genre_text)
  cosine_simi = cosine_similarity(tfidf_matrix, tfidf_matrix)
  title_to_idx = pd.Series(movies.index, index = movies['title'])

  # 4. Get the indices of similar movies to those rated by the user based on cosine similarity. Keep only the top 5.
  all_sims = []
  for title in rated:
    idx = title_to_idx[title]
    sim_scores = list(enumerate(cosine_simi[idx]))
    sim_scores = sorted(sim_scores, key = lambda x : x[1], reverse = True)
    top_idxs = [i for i, score in sim_scores[1:1+top_n]]
    all_sims.extend(top_idxs)

  # 5. Remove duplicates and movies already rated by the user.
  unique_idxs = []
  for i in all_sims:
    if i not in unique_idxs:
      unique_idxs.append(i)
  rec_idxs = [i for i in unique_idxs if movies.at[i, 'title'] not in rated]
  return movies.loc[rec_idxs, 'title'].head(top_n).tolist()

# Part 3

**Write a function that takes in a user id and the dataframe you created before that contains 'user_id', 'title', and 'rating'. The function should return collaborative filtering recommendations for this user based on a user-item interaction matrix. Here are steps you can take:**


1.   Create the user-item matrix using Pandas' pivot_table.
2.   Fill missing values with zeros in this matrix.
3. Calculate user-user similarity matrix using cosine similarity.
4. Get the array of similarity scores of the target user with all other users from the similarity matrix.
5. Extract, say the the top 5 most similar users (excluding the target user).
6. Generate movie recommendations based on the most similar users.
7. Remove duplicate movies recommendations.


In [45]:
def collaborative_filtering_recommendations(user_id, user_movie_ratings, top_n = 5):
  # 1. Create the user-item matrix using Pandas' pivot_table.
  user_item = user_movie_rating.pivot_table(index = 'user_id', columns = 'title', values = 'rating')

  # 2. Fill missing values with zeros in this matrix.
  user_item_filled = user_item.fillna(0)

  # 3. Calculate user-user similarity matrix using cosine similarity.
  sim_matrix = cosine_similarity(user_item_filled)
  sim_df = pd.DataFrame(sim_matrix, index = user_item_filled.index, columns = user_item_filled.index)

  # 4. Get the array of similarity scores of the target user with all other users from the similarity matrix.
  user_sims = sim_df[user_id].drop(labels = [user_id])

  # 5. Extract, say the the top 5 most similar users (excluding the target user).
  top_users = user_sims.sort_values(ascending = False).head(top_n).index

  # 6. Generate movie recommendations based on the most similar users.
  target_ratings = user_item_filled.loc[user_id]
  seen = set(target_ratings[target_ratings > 0].index)
  scores = {}
  for other in top_users:
    sim_score = user_sims[other]
    other_ratings = user_item_filled.loc[other]
    for title, rating in other_ratings.items():
      if title in seen or rating == 0:
        continue
      scores.setdefault(title, 0.0)
      scores[title]+= rating * sim_score

  # 7. Remove duplicate movies recommendations.
  recommended = sorted(scores.items(), key = lambda x : x[1], reverse = True)
  return [title for title, score in recommended[:top_n]]

# Part 4

**Now, test your recommendations engines! Select a few user ids and generate recommendations using both functions you've written. Are the recommendations similar? Do the recommendations make sense?**

In [53]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import pandas as pd

full_genre_cols = [
    "genre_unknown","Action","Adventure","Animation","Children","Comedy",
    "Crime","Documentary","Drama","Fantasy","Film-Noir","Horror",
    "Musical","Mystery","Romance","Sci-Fi","Thriller","War","Western"
]

col_names = ['movie_id','title','release_date','video_release_date','imdb_url'] + full_genre_cols

movies = pd.read_csv(
    'ml-100k/u.item',
    sep='|',
    names=col_names,
    encoding='latin-1',
    dtype={'movie_id': 'int64'} )

user_movie_ratings = (ratings.merge(movies[['movie_id','title']], on='movie_id', how='inner')[['user_id','title','rating']])

def content_based_recommendations(user_id, user_movie_ratings, movies, genre_cols, top_n=5):
    rated_titles = user_movie_ratings.loc[user_movie_ratings['user_id'] == user_id, 'title'].tolist()

    genre_text = movies[genre_cols].apply(lambda row: " ".join([g for g in genre_cols if row[g] == 1]), axis=1)

    tfidf = TfidfVectorizer()
    tfidf_matrix = tfidf.fit_transform(genre_text)
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

    title_to_idx = {title: idx for idx, title in enumerate(movies['title'])}

    all_sims = []
    for title in rated_titles:
        idx = title_to_idx[title]
        sim_scores = list(enumerate(cosine_sim[idx]))
        top_idxs = [i for i, _ in sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]]
        all_sims.extend(top_idxs)

    seen = set(rated_titles)
    rec_idxs = []
    for i in all_sims:
        t = movies.at[i,'title']
        if t not in seen and i not in rec_idxs:
            rec_idxs.append(i)
    return movies.loc[rec_idxs, 'title'].head(top_n).tolist()

def collaborative_filtering_recommendations(user_id, user_movie_ratings, top_n=5):
    user_item = user_movie_ratings.pivot_table(index='user_id', columns='title', values='rating').fillna(0)


    sim_matrix = cosine_similarity(user_item)
    sim_df = pd.DataFrame(sim_matrix,
                          index=user_item.index,
                          columns=user_item.index)

    user_sims = sim_df[user_id].drop(labels=[user_id])
    top_users = user_sims.nlargest(top_n).index


    seen = set(user_item.loc[user_id].loc[lambda x: x>0].index)
    scores = {}
    for other in top_users:
        w = user_sims[other]
        for title, rating in user_item.loc[other].items():
            if rating > 0 and title not in seen:
                scores[title] = scores.get(title, 0) + rating * w


    recs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [title for title,_ in recs[:top_n]]


test_users = range(1,10)
for uid in test_users:
    cb = content_based_recommendations(uid, user_movie_ratings, movies, genre_cols, top_n=5)
    cf = collaborative_filtering_recommendations(uid, user_movie_ratings, top_n=5)
    print(f"\nUser {uid} --> \n  CB: {cb}\n  CF: {cf}")



User 1 --> 
  CB: ['Faust (1994)', 'Close Shave, A (1995)', 'Speed (1994)', 'Saint, The (1997)', 'Tomorrow Never Dies (1997)']
  CF: ['Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963)', 'Casablanca (1942)', 'Stand by Me (1986)', 'Heathers (1989)', 'Piano, The (1993)']

User 2 --> 
  CB: ['Dead Man Walking (1995)', "Mr. Holland's Opus (1995)", 'White Balloon, The (1995)', 'Belle de jour (1967)', 'Brothers McMullen, The (1995)']
  CF: ['Amistad (1997)', 'Lone Star (1996)', 'Michael Collins (1996)', 'Spitfire Grill, The (1996)', 'Game, The (1997)']

User 3 --> 
  CB: ['Birdcage, The (1996)', 'Brothers McMullen, The (1995)', 'To Wong Foo, Thanks for Everything! Julie Newmar (1995)', 'Billy Madison (1995)', 'Clerks (1994)']
  CF: ['Titanic (1997)', 'Apt Pupil (1998)', 'Amistad (1997)', 'English Patient, The (1996)', 'As Good As It Gets (1997)']

User 4 --> 
  CB: ['Lawnmower Man 2: Beyond Cyberspace (1996)', 'Unforgettable (1996)', 'Island of Dr. Moreau, The (199

There is generally little overlap between the two sets except for a few big hits. Both make sense on their own-content-based sticks to each user's preferred genres, while collaborative filtering surfaces what like-minded viewers enjoyed.